# GCN Cloud Notebook: Richer + Graph Features Multi-Seed

This notebook is configured for Google Colab using GitHub as the code source. It clones the repo into `/content`, keeps Google Drive optional, and validates the current best GCN setup across multiple random seeds.

This standalone notebook includes the best prior confirmed changes:
- Adam with the baseline two-phase schedule
- weighted edges from `adjacency_area.csv`
- richer node features derived at load time
- graph-level summary features concatenated after graph pooling

No new lattice sets are required.

In [ ]:
import sys

IN_COLAB = 'google.colab' in sys.modules
print(f'Running in Colab: {IN_COLAB}')

if IN_COLAB:
    %pip -q install torch-geometric
else:
    print('Colab dependency install cell skipped.')

In [ ]:
REPO_URL = 'https://github.com/aadams2006/NSF-REU-Summer-26.git'
REPO_DIR = '/content/NSF-REU-Summer-26'

if IN_COLAB:
    import os
    if not os.path.isdir(REPO_DIR):
        !git clone {REPO_URL} {REPO_DIR}
    else:
        print(f'Repo already exists at {REPO_DIR}')
else:
    print('Git clone cell skipped outside Colab.')

In [ ]:
from datetime import datetime
from getpass import getpass
from pathlib import Path
import os
import sys

USE_DRIVE_FOR_DATA = False
SAVE_OUTPUTS_TO_DRIVE = True
PUSH_RESULTS_TO_GITHUB = False
PUSH_MODEL_TO_GITHUB = False
DRIVE_DATA_ROOT = '/content/drive/MyDrive/lattice_data'
DRIVE_OUTPUT_ROOT = '/content/drive/MyDrive/GCN_Cloud_Outputs_Richer_Graph_Features_Multi_Seed'
GIT_RESULTS_SUBDIR = 'active_projects/voronoi_lattice_pipeline/gnn_prototype/GCN_Cloud_Outputs_Richer_Graph_Features_Multi_Seed'
GIT_BRANCH = 'main'
GITHUB_TOKEN = os.environ.get('GITHUB_TOKEN', '').strip()
GIT_COMMIT_USERNAME = os.environ.get('GIT_COMMIT_USERNAME', '').strip()
GIT_COMMIT_EMAIL = os.environ.get('GIT_COMMIT_EMAIL', '').strip()
RUN_STAMP = datetime.now().strftime('%Y%m%d_%H%M%S')

if IN_COLAB:
    repo_root = Path(REPO_DIR).resolve()
else:
    repo_root = Path.cwd().resolve()

pipeline_root = repo_root / 'active_projects' / 'voronoi_lattice_pipeline'
module_dir = pipeline_root / 'gnn_prototype'
if not (module_dir / 'colab_gnn_stiffness_prototype.py').is_file():
    raise FileNotFoundError(f'Module not found at {module_dir}')

if str(module_dir) not in sys.path:
    sys.path.insert(0, str(module_dir))

if IN_COLAB and (USE_DRIVE_FOR_DATA or SAVE_OUTPUTS_TO_DRIVE):
    from google.colab import drive
    drive.mount('/content/drive')

if USE_DRIVE_FOR_DATA:
    if not IN_COLAB:
        raise RuntimeError('USE_DRIVE_FOR_DATA is only supported in Colab.')
    drive_data_root = Path(DRIVE_DATA_ROOT)
    train_root = drive_data_root / 'Randomness_Sweep'
    predict_root = drive_data_root / 'Lattice_Guess_Prediction_Input_Data'
else:
    train_root = pipeline_root / 'source_archives' / 'lattice_data' / 'Randomness_Sweep'
    predict_root = pipeline_root / 'datasets' / 'Lattice_Guess_Prediction_Input_Data'

if IN_COLAB and SAVE_OUTPUTS_TO_DRIVE:
    output_root = Path(DRIVE_OUTPUT_ROOT)
else:
    output_root = Path('/content/gnn_outputs_richer_graph_features_multi_seed') if IN_COLAB else pipeline_root / 'gnn_prototype' / 'outputs_richer_graph_features_multi_seed'

git_output_root = repo_root / GIT_RESULTS_SUBDIR
output_dir = output_root / f'run_{RUN_STAMP}'
output_dir.mkdir(parents=True, exist_ok=True)
(output_root / 'latest_run.txt').write_text(str(output_dir), encoding='utf-8')

if PUSH_RESULTS_TO_GITHUB:
    if not GITHUB_TOKEN:
        GITHUB_TOKEN = getpass('Enter GitHub token: ').strip()
    if not GIT_COMMIT_USERNAME:
        GIT_COMMIT_USERNAME = input('Enter Git commit username or display name: ').strip()
    if not GIT_COMMIT_EMAIL:
        GIT_COMMIT_EMAIL = input('Enter Git commit email (GitHub noreply or verified email): ').strip()

print(f'Repo root: {repo_root}')
print(f'Pipeline root: {pipeline_root}')
print(f'Train data: {train_root}')
print(f'Prediction data: {predict_root}')
print(f'Output root: {output_root}')
print(f'Current run dir: {output_dir}')
print(f'Git output root: {git_output_root}')
print(f'Push results to GitHub: {PUSH_RESULTS_TO_GITHUB}')
print(f'GitHub token loaded: {bool(GITHUB_TOKEN)}')
print(f'Git commit username loaded: {bool(GIT_COMMIT_USERNAME)}')
print(f'Git commit email loaded: {bool(GIT_COMMIT_EMAIL)}')

In [ ]:
import json
import matplotlib.pyplot as plt
import pandas as pd
import shutil
import subprocess
from IPython.display import display

from colab_gnn_stiffness_prototype import (
    TrainingConfig,
    SimpleGNN,
    create_data_loaders,
    evaluate_model,
    load_lattice_dataset,
    normalize_feature_splits,
    predict_on_directory,
    save_run_artifacts,
    set_seed,
    split_dataset,
    train_model,
)


def run_single_seed(seed: int, output_dir: Path) -> dict:
    config = TrainingConfig(seed=seed)
    set_seed(config.seed)

    dataset = load_lattice_dataset(train_root)
    train_data, val_data, test_data = split_dataset(dataset, seed=config.seed)
    scaler = normalize_feature_splits(train_data, val_data, test_data)
    train_loader, val_loader, test_loader = create_data_loaders(
        train_data,
        val_data,
        test_data,
        batch_size=config.batch_size,
    )

    model = SimpleGNN(
        input_dim=train_data[0].x.shape[1],
        hidden_dim=config.hidden_dim,
        graph_feature_dim=train_data[0].graph_attr.shape[1],
    )
    history = train_model(model, train_loader, val_loader, config)

    metrics_by_split = {}
    for split_name, loader in (("Train", train_loader), ("Validation", val_loader), ("Test", test_loader)):
        _, _, metrics = evaluate_model(model, loader, device=config.device)
        metrics_by_split[split_name] = metrics

    prediction_results, prediction_metrics = predict_on_directory(
        model,
        predict_root,
        scaler,
        device=config.device,
    )

    seed_dir = output_dir / f'seed_{seed}'
    save_run_artifacts(
        seed_dir,
        model,
        scaler,
        history,
        metrics_by_split,
        prediction_results=prediction_results,
    )

    run_summary = {
        'seed': seed,
        'feature_mode': 'weighted_edges_plus_richer_node_features_plus_graph_summary_features',
        'input_dim': int(train_data[0].x.shape[1]),
        'graph_feature_dim': int(train_data[0].graph_attr.shape[1]),
        'epochs_completed': int(history['epochs_completed']),
        'best_val_loss': float(history['best_val_loss']),
        'prediction_metrics': prediction_metrics,
    }
    (seed_dir / 'run_config.json').write_text(json.dumps(run_summary, indent=2), encoding='utf-8')

    return {
        'summary_row': {
            'Seed': seed,
            'Input_Dim': int(train_data[0].x.shape[1]),
            'Graph_Feature_Dim': int(train_data[0].graph_attr.shape[1]),
            'Epochs_Completed': int(history['epochs_completed']),
            'Best_Val_Loss': float(history['best_val_loss']),
            'Validation_RMSE': metrics_by_split['Validation']['RMSE'],
            'Validation_MAE': metrics_by_split['Validation']['MAE'],
            'Validation_R2': metrics_by_split['Validation']['R2'],
            'Test_RMSE': metrics_by_split['Test']['RMSE'],
            'Test_MAE': metrics_by_split['Test']['MAE'],
            'Test_R2': metrics_by_split['Test']['R2'],
            'Prediction_RMSE': prediction_metrics['RMSE'],
            'Prediction_MAE': prediction_metrics['MAE'],
            'Prediction_R2': prediction_metrics['R2'],
        },
        'save_dir': seed_dir,
    }


def build_aggregate_frame(summary_frame: pd.DataFrame) -> pd.DataFrame:
    metric_columns = [
        'Best_Val_Loss',
        'Validation_RMSE',
        'Validation_MAE',
        'Validation_R2',
        'Test_RMSE',
        'Test_MAE',
        'Test_R2',
        'Prediction_RMSE',
        'Prediction_MAE',
        'Prediction_R2',
    ]
    return pd.DataFrame(
        {
            'mean': summary_frame[metric_columns].mean(),
            'std': summary_frame[metric_columns].std(ddof=1),
            'min': summary_frame[metric_columns].min(),
            'max': summary_frame[metric_columns].max(),
        }
    )


def plot_multi_seed_summary(summary_frame: pd.DataFrame, save_path: Path | None = None) -> None:
    ordered = summary_frame.sort_values('Seed')
    fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

    axes[0].plot(ordered['Seed'], ordered['Validation_R2'], marker='o', linewidth=2)
    axes[0].plot(ordered['Seed'], ordered['Test_R2'], marker='o', linewidth=2)
    axes[0].set_title('R2 by Seed')
    axes[0].set_xlabel('Seed')
    axes[0].set_ylabel('R2')
    axes[0].grid(alpha=0.3)
    axes[0].legend(['Validation', 'Test'])

    axes[1].plot(ordered['Seed'], ordered['Validation_RMSE'], marker='o', linewidth=2)
    axes[1].plot(ordered['Seed'], ordered['Test_RMSE'], marker='o', linewidth=2)
    axes[1].set_title('RMSE by Seed')
    axes[1].set_xlabel('Seed')
    axes[1].set_ylabel('RMSE')
    axes[1].grid(alpha=0.3)
    axes[1].legend(['Validation', 'Test'])

    axes[2].plot(ordered['Seed'], ordered['Prediction_R2'], marker='o', linewidth=2)
    axes[2].set_title('Prediction R2 by Seed')
    axes[2].set_xlabel('Seed')
    axes[2].set_ylabel('Prediction R2')
    axes[2].grid(alpha=0.3)

    plt.tight_layout()
    if save_path is not None:
        fig.savefig(save_path, dpi=200, bbox_inches='tight')
    plt.show()


In [ ]:
SEEDS = (11, 42, 73, 101, 202)
print(f'Seeds: {SEEDS}')
print('Model: Adam baseline schedule with weighted edges + richer node features + graph-level summary features from existing lattice files')

In [ ]:
experiment_results = {}
summary_rows = []

for seed in SEEDS:
    print(f'=== Running seed {seed} ===')
    result = run_single_seed(seed, output_dir)
    experiment_results[seed] = result
    summary_rows.append(result['summary_row'])

summary_frame = pd.DataFrame(summary_rows).sort_values('Seed').reset_index(drop=True)
summary_path = output_dir / 'multi_seed_summary.csv'
summary_frame.to_csv(summary_path, index=False)

print(f'Saved per-seed summary to {summary_path}')
display(summary_frame)

In [ ]:
aggregate_frame = build_aggregate_frame(summary_frame)
aggregate_path = output_dir / 'multi_seed_aggregate.csv'
aggregate_frame.to_csv(aggregate_path)

print(f'Saved aggregate summary to {aggregate_path}')
display(aggregate_frame)

In [ ]:
summary_plot_path = output_dir / 'multi_seed_metric_summary.png'
plot_multi_seed_summary(summary_frame, save_path=summary_plot_path)
print(f'Saved multi-seed plot to {summary_plot_path}')

In [ ]:
best_seed = int(summary_frame.sort_values('Test_RMSE').iloc[0]['Seed'])
best_result = experiment_results[best_seed]

print(f'Best seed by Test RMSE: {best_seed}')
display(summary_frame[summary_frame['Seed'] == best_seed])

In [ ]:
print(f'Parent run dir: {output_dir}')
parent_files = sorted(path.name for path in output_dir.iterdir() if path.is_file())
print('Parent-level files:')
for file_name in parent_files:
    print(f' - {file_name}')

for seed, result in experiment_results.items():
    print(f' - seed {seed}: {result["save_dir"]}')

if PUSH_RESULTS_TO_GITHUB:
    if not IN_COLAB:
        raise RuntimeError('GitHub auto-push is only intended for the Colab clone workflow.')
    if not GITHUB_TOKEN:
        raise ValueError('Set GITHUB_TOKEN before enabling PUSH_RESULTS_TO_GITHUB.')

    git_run_dir = git_output_root / output_dir.name
    if git_run_dir.exists():
        shutil.rmtree(git_run_dir)
    shutil.copytree(output_dir, git_run_dir)
    if not PUSH_MODEL_TO_GITHUB:
        for model_path in git_run_dir.rglob('lattice_gnn_model.pt'):
            model_path.unlink()

    (git_output_root / 'latest_run.txt').write_text(str(git_run_dir.relative_to(repo_root)), encoding='utf-8')

    subprocess.run(['git', '-C', str(repo_root), 'config', 'user.name', GIT_COMMIT_USERNAME], check=True)
    subprocess.run(['git', '-C', str(repo_root), 'config', 'user.email', GIT_COMMIT_EMAIL], check=True)

    remote_url = subprocess.run(
        ['git', '-C', str(repo_root), 'remote', 'get-url', 'origin'],
        check=True,
        capture_output=True,
        text=True,
    ).stdout.strip()
    auth_url = remote_url.replace('https://', f'https://{GITHUB_TOKEN}@', 1)
    subprocess.run(['git', '-C', str(repo_root), 'remote', 'set-url', 'origin', auth_url], check=True)

    try:
        subprocess.run(['git', '-C', str(repo_root), 'add', str(git_run_dir), str(git_output_root / 'latest_run.txt')], check=True)
        diff_result = subprocess.run(
            ['git', '-C', str(repo_root), 'diff', '--cached', '--quiet'],
            check=False,
        )
        if diff_result.returncode == 0:
            print('No GitHub changes to commit.')
        else:
            commit_message = f'Add richer-graph-feature multi-seed GCN cloud results for {output_dir.name}'
            subprocess.run(['git', '-C', str(repo_root), 'commit', '-m', commit_message], check=True)
            subprocess.run(['git', '-C', str(repo_root), 'push', 'origin', GIT_BRANCH], check=True)
            print(f'Pushed results to GitHub under {git_run_dir.relative_to(repo_root)}')
    finally:
        subprocess.run(['git', '-C', str(repo_root), 'remote', 'set-url', 'origin', remote_url], check=True)

In [ ]:
if IN_COLAB and not SAVE_OUTPUTS_TO_DRIVE:
    from google.colab import files
    archive_path = '/content/gnn_outputs_richer_graph_features_multi_seed.zip'
    !cd /content && zip -qr gnn_outputs_richer_graph_features_multi_seed.zip gnn_outputs_richer_graph_features_multi_seed
    files.download(archive_path)
else:
    print(f'Outputs are in {output_dir}')